Zueri wie neu Project

In [67]:
import pandas as pd 
import geopandas as gpd

Reading the ZüriWieNeu data layer. This is done using Pandas when there is no spatial component but only tabular data.

In [68]:
data_zwn = pd.read_csv("../data/raw/stzh.zwn_meldungen_p.csv")

Creating a GeoDataFrame to represent the coordinates as  geometries rather than a strings because they were originally stored as WKT geometries for each report (Lecture 7.1). After you can check and change the crs to make it the same as the Quartiere Layer later on. This step prepares the dataset for joining. It is required whenever spatial realtionships become relevant. 

In [69]:
zwn_gdf = gpd.GeoDataFrame (
    data_zwn, 
    geometry=gpd.points_from_xy(data_zwn["e"], data_zwn["n"]),
    crs="EPSG:2056"
    )


The datetime must also be transformed because in zwn_gdf it is initially stored as a string (Lecture 6.7). After the conversion to a datetime format two new columns are created (year, month).

In [70]:
zwn_gdf["requested_datetime"] = pd.to_datetime(zwn_gdf["requested_datetime"])
zwn_gdf["agency_sent_datetime"] = pd.to_datetime(zwn_gdf["agency_sent_datetime"])
zwn_gdf["updated_datetime"] = pd.to_datetime(zwn_gdf["updated_datetime"])
zwn_gdf["year_requested"] = zwn_gdf["requested_datetime"].dt.year
zwn_gdf["month_requested"] = zwn_gdf["requested_datetime"].dt.month



Now its time to load and prep the Quartiere Data. Two layers are used. One conatins the coordinates/geometries of each Quartier and the other contains the attributes of each Quartier such as their names etc.  

In [71]:
quartiere_geom = gpd.read_file("../data/raw/stzh.adm_statistische_quartiere_map.json")
quartiere_attr = gpd.read_file("../data/raw/stzh.adm_statistische_quartiere_b_p.json")


Since both datasets contain relevant information, they are merged (Lecture 6.4). Because both datasets include a geometry attriubt, the resulting GeoDataFrame contains two geometry columns. ONe represents the active geometry, while the other is incomplete or unnecessary. Teerefore, the main geometry column is selected and renamed while the other one is dropped. This step results in a clean Quartiere layer with valid polygon geometries and all required attributes. 

In [72]:
quartiere = quartiere_geom.merge(quartiere_attr, on="objectid")

quartiere = quartiere.rename(columns={"geometry_x": "geometry"})
quartiere = quartiere.set_geometry("geometry")
quartiere = quartiere.drop(columns=["geometry_y", "objid", "ori", "hali", "vali"])



To be able to join the Quartiere data it with the ZüriWieNeu data, both datasets must share the same coordinate system. Therefore, the Quartiere dataset is reprojected to EPSG:2056.

In [73]:
quartiere = quartiere.to_crs(epsg=2056)


Now comes the crucial step of joining the preped Quartiere layer with the ZüriWieNeu dataset (Lecture 7.6). Using the predicate "within" each point from the zwn dataset is assigned to the Quartier in which it is located.

In [74]:
zwn_quertiere_joined = gpd.sjoin(zwn_gdf, quartiere, predicate="within")
